# 04b — Social Media Charts (Altair + vl-convert)

Publication-ready PNG charts using the @unwelcomedata brand palette.
All charts export to twitter_landscape (1600×900px) with watermark.

**4 Production Charts:**
1. National Abortion Comparison (side-by-side: without vs. with)
2. Top 10 Causes by Sex (stacked bars: male vs. female)
3. Abortion Impact by Race (White)
4. Abortion Impact by Race (Black/African American)

In [ ]:
import sys
import os
from pathlib import Path

import pandas as pd
import duckdb
import yaml
import altair as alt

# Find project root
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT.parent / 'shared'))

from src.viz_social import save_social
from viz import PRESETS, SEX_COLORS, PALETTE

with open(PROJECT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

# Create outputs/social directory if needed
social_dir = PROJECT / 'outputs' / 'social'
social_dir.mkdir(parents=True, exist_ok=True)

# Connect to DuckDB
conn = duckdb.connect(str(PROJECT / 'data' / 'project.duckdb'))

# Get total abortions (national_total measure, 2024)
abort_total = conn.execute('''
  SELECT value
  FROM abortions
  WHERE measure = 'national_total' AND year = 2024
''').df()['value'].iloc[0]


# Display names for cleaner labels
DISPLAY_NAMES = {
    'Diseases of heart': 'Heart disease',
    'Malignant neoplasms': 'Cancer',
    'Chronic lower respiratory diseases': 'Respiratory disease',
    'Cerebrovascular diseases': 'Stroke',
    'Alzheimer disease': "Alzheimer's",
    'Diabetes mellitus': 'Diabetes',
    'Accidents (unintentional injuries)': 'Accidents',
    'Intentional self-harm (suicide)': 'Suicide',
    'Chronic liver disease and cirrhosis': 'Liver disease',
    'Nephritis, nephrotic syndrome and nephrosis': 'Kidney disease',
    'Influenza and pneumonia': 'Flu/Pneumonia',
    'Essential hypertension and hypertensive renal disease': 'Hypertension',
    'Assault (homicide)': 'Homicide',
    'Pregnancy, childbirth and the puerperium': 'Pregnancy/childbirth',
}

def short_name(cause: str) -> str:
    """Map verbose cause name to short display name."""
    return DISPLAY_NAMES.get(cause, cause)

print('✓ Environment loaded')
print(f'✓ Social charts will export to: {social_dir}')
print(f'✓ Total abortions 2024: {abort_total:,.0f}')

## Chart 1: National Abortion Comparison

"If abortion were a cause of death, it would rank as the #2-3 leading cause in the US"

In [ ]:
# Get top 10 causes nationally
top_10_national = conn.execute('''
  SELECT 
    COALESCE(SUBSTR(cause, 2), cause) as cause_clean,
    deaths
  FROM mortality_national
  ORDER BY deaths DESC
  LIMIT 10
''').df()

# Apply display names
top_10_national['cause_display'] = top_10_national['cause_clean'].apply(short_name)

# Build without abortion (sorted descending - largest at top)
df_without = top_10_national[['cause_display', 'deaths']].copy()
df_without.columns = ['cause', 'deaths']
df_without['comparison'] = 'Without Abortion'
df_without['is_abortion'] = False

# Build with abortion (reranked with abortion at position 2)
df_with_list = []
rank = 0
for idx, row in top_10_national.iterrows():
    if rank == 1 and abort_total > row['deaths']:
        df_with_list.append({'cause': 'Induced Abortion', 'deaths': abort_total, 'comparison': 'With Abortion', 'is_abortion': True})
        rank += 1
    df_with_list.append({'cause': row['cause_display'], 'deaths': row['deaths'], 'comparison': 'With Abortion', 'is_abortion': False})
    rank += 1
    if rank > 11:
        break

df_with = pd.DataFrame(df_with_list)

# Combine
df_combined = pd.concat([df_without, df_with], ignore_index=True)

# Create sorted order (descending, largest at top)
cause_order = df_without.sort_values('deaths', ascending=True)['cause'].tolist()

print(f'Chart 1 data ready: {len(df_without)} causes without abortion')

In [ ]:
# Prepare data for Chart 1
# First, get sex breakdown for all causes in df_combined
sex_by_cause = conn.execute(f'''
  SELECT 
    COALESCE(SUBSTR(icd_10_113_cause_list, 2), icd_10_113_cause_list) as cause_raw,
    sex,
    SUM(deaths) as deaths
  FROM mortality_sex_age
  WHERE icd_10_113_cause_list IN ({','.join([f"'{c}'" for c in top_10_national['cause_clean']])})
  GROUP BY icd_10_113_cause_list, sex
''').df()

# Create a mapping of cause_display to male/female breakdown
sex_pcts = {}
for cause_display in df_without['cause']:
    # Find the cause_clean that maps to this display name
    cause_clean = None
    for idx, row in top_10_national.iterrows():
        if row['cause_display'] == cause_display:
            cause_clean = row['cause_clean']
            break
    
    if cause_clean:
        male_deaths = sex_by_cause[(sex_by_cause['cause_raw'] == cause_clean) & (sex_by_cause['sex'] == 'Male')]['deaths'].sum()
        female_deaths = sex_by_cause[(sex_by_cause['cause_raw'] == cause_clean) & (sex_by_cause['sex'] == 'Female')]['deaths'].sum()
        total = male_deaths + female_deaths
        if total > 0:
            sex_pcts[cause_display] = {
                'male_pct': round(100 * male_deaths / total),
                'female_pct': round(100 * female_deaths / total),
                'total': int(total)
            }

# Add sex percentages to df_combined
df_combined['male_pct'] = df_combined['cause'].apply(lambda c: sex_pcts.get(c, {}).get('male_pct', 0))
df_combined['female_pct'] = df_combined['cause'].apply(lambda c: sex_pcts.get(c, {}).get('female_pct', 0))
df_combined['total_display'] = df_combined['cause'].apply(lambda c: sex_pcts.get(c, {}).get('total', int(df_combined[df_combined['cause']==c]['deaths'].sum())))

# Fix "Induced Abortion" to just "Abortion"
df_combined['cause'] = df_combined['cause'].replace('Induced Abortion', 'Abortion')

# Resort by deaths descending (largest at top)
df_combined = df_combined.sort_values('deaths', ascending=True).reset_index(drop=True)
cause_order = df_combined['cause'].tolist()

print(f'Chart 1 data prepared: {len(df_combined)} rows')
print('Sample row:', df_combined.iloc[0].to_dict())

In [ ]:
# Build Chart 1: Horizontal stacked bars with sex percentages
# Reshape for stacking: need male/female breakdown per cause
df_male = df_combined[['cause', 'deaths', 'comparison', 'male_pct', 'total_display']].copy()
df_male['sex'] = 'Male'
df_male['sex_deaths'] = df_combined['deaths'] * (df_combined['male_pct'] / 100)

df_female = df_combined[['cause', 'deaths', 'comparison', 'female_pct', 'total_display']].copy()
df_female['sex'] = 'Female'
df_female['sex_deaths'] = df_combined['deaths'] * (df_combined['female_pct'] / 100)

df_stacked = pd.concat([df_male, df_female], ignore_index=True)

# Build stacked bars with sex breakdown
bars = alt.Chart(df_stacked).mark_bar().encode(
    y=alt.Y('cause:N', title='', sort=cause_order, 
           axis=alt.Axis(labels=True, domain=False, ticks=False, labelFontSize=11)),
    x=alt.X('sex_deaths:Q', title='', axis=None, stack='zero'),
    color=alt.Color('sex:N',
        scale=alt.Scale(domain=['Male', 'Female'], range=['#005F73', '#AE2012']),
        legend=alt.Legend(title=None, orient='top', labelFontSize=11)
    ),
    order=alt.Order('sex:N', sort='ascending'),
).properties(
    width=1450,
    height=720,
    title={
        'text': 'If Abortion Were Counted as a Cause of Death',
        'subtitle': 'It would rank as the #2-3 leading cause in the US (2024)',
        'anchor': 'start',
        'offset': 10,
    }
)

# Text labels: percentages inside bars
text_pct = alt.Chart(df_combined).mark_text(align='center', color='white', fontSize=9, fontWeight='bold').encode(
    y=alt.Y('cause:N', sort=cause_order),
    x=alt.X('deaths:Q', stack='zero'),
    text=alt.condition(
        alt.datum.male_pct > 0,
        alt.Text('male_pct:Q', format='d'),
        alt.value('')
    )
)

# Text labels: total count to right of bars
text_total = alt.Chart(df_combined).mark_text(align='left', dx=5, fontSize=10, fontColor='#374151').encode(
    y=alt.Y('cause:N', sort=cause_order),
    x=alt.X('deaths:Q'),
    text=alt.Text('total_display:Q', format=',')
)

chart1 = (bars + text_pct + text_total).configure_axis(
    grid=False,
    domain=False,
    labelColor='#374151'
).configure_view(
    strokeWidth=0
)

print('Chart 1 created with sex breakdown and totals')
chart1

In [ ]:
# Export Chart 1
save_social(chart1, cfg, '01_abortion_comparison_national', preset='twitter_landscape')
print('✓ Chart 1 exported')

## Chart 2: Top 10 Causes by Sex

In [ ]:
# Get top 10 causes from national table
top_causes = conn.execute('''
  SELECT cause
  FROM mortality_national
  ORDER BY deaths DESC
  LIMIT 10
''').df()['cause'].tolist()

# Get sex breakdown for each
sex_breakdown = conn.execute(f'''
  SELECT 
    COALESCE(SUBSTR(icd_10_113_cause_list, 2), icd_10_113_cause_list) as cause,
    sex,
    SUM(deaths) as deaths
  FROM mortality_sex_age
  WHERE icd_10_113_cause_list IN ({','.join([f"'{c}'" for c in top_causes])})
  GROUP BY icd_10_113_cause_list, sex
  ORDER BY icd_10_113_cause_list, sex
''').df()

# Pivot and sort
sex_pivot = sex_breakdown.pivot_table(
    index='cause', columns='sex', values='deaths', aggfunc='sum'
).reset_index()

# Ensure both columns exist
sex_pivot['Female'] = sex_pivot.get('Female', 0)
sex_pivot['Male'] = sex_pivot.get('Male', 0)

sex_pivot['total'] = sex_pivot['Female'] + sex_pivot['Male']
sex_pivot = sex_pivot.sort_values('total', ascending=True).reset_index(drop=True)

# Melt for stacked bar
sex_long = sex_pivot[['cause', 'Female', 'Male']].melt(
    id_vars=['cause'],
    value_vars=['Female', 'Male'],
    var_name='sex',
    value_name='deaths'
)

print(f'Sex breakdown ready: {len(sex_long)} rows')

In [ ]:
# Build Chart 2: Stacked bars
color_map = {'Male': SEX_COLORS.get('Male', '#005F73'), 'Female': SEX_COLORS.get('Female', '#AE2012')}

chart2 = alt.Chart(sex_long).mark_bar().encode(
    y=alt.Y('cause:N', title='', sort=sex_pivot['cause'].tolist()),
    x=alt.X('deaths:Q', title='Deaths (2024)', stack='zero'),
    color=alt.Color('sex:N', scale=alt.Scale(
        domain=['Female', 'Male'],
        range=[color_map['Female'], color_map['Male']]
    ), title='Sex'),
    tooltip=['cause', 'sex', 'deaths'],
).properties(
    width=1450,
    height=720,
    title={
        'text': 'Leading Causes of Death by Sex (2024)',
        'subtitle': 'Top 10 causes nationally, broken down by male and female deaths',
        'anchor': 'start',
        'offset': 10,
    }
).configure_axis(
    labelFontSize=10,
    titleFontSize=11,
    labelColor='#374151',
    titleColor='#374151'
).configure_legend(
    labelFontSize=11,
    orient='top',
)

print('Chart 2 created')
chart2

In [ ]:
# Export Chart 2
save_social(chart2, cfg, '02_top_10_causes_by_sex', preset='twitter_landscape')
print('✓ Chart 2 exported')

## Charts 3 & 4: Abortion by Race

In [ ]:
races_to_chart = ['White', 'Black or African American']
charts_by_race = {}

for idx, race in enumerate(races_to_chart, 1):
    print(f'Building Chart {2+idx}: {race}...')
    
    # Get top 10 causes for this race
    top_10_race = conn.execute(f'''
      SELECT 
        COALESCE(SUBSTR(icd_10_113_cause_list, 2), icd_10_113_cause_list) as cause,
        SUM(deaths) as deaths
      FROM mortality_race_sex
      WHERE single_race_6 = '{race}'
      GROUP BY icd_10_113_cause_list
      ORDER BY deaths DESC
      LIMIT 10
    ''').df()
    
    df_without_race = top_10_race.copy()
    df_without_race['comparison'] = 'Without Abortion'
    
    # Create with abortion
    df_with_race_list = []
    rank = 1
    for row_idx, row_tuple in enumerate(top_10_race.itertuples(), 1):
        if rank <= 2 and abort_total > row_tuple.deaths:
            df_with_race_list.append({'cause': 'Induced Abortion', 'deaths': abort_total, 'comparison': 'With Abortion'})
            rank += 1
        df_with_race_list.append({'cause': row_tuple.cause, 'deaths': row_tuple.deaths, 'comparison': 'With Abortion'})
        rank += 1
        if len(df_with_race_list) >= 11:
            break
    
    df_with_race = pd.DataFrame(df_with_race_list)
    df_race_combined = pd.concat([df_without_race, df_with_race], ignore_index=True)
    
    cause_order_race = df_without_race.sort_values('deaths', ascending=True)['cause'].tolist()
    
    # Build chart
    chart = alt.Chart(df_race_combined).mark_bar().encode(
        y=alt.Y('cause:N', title='', sort=cause_order_race),
        x=alt.X('deaths:Q', title='Deaths (2024)'),
        color=alt.Color('comparison:N', scale=alt.Scale(
            domain=['Without Abortion', 'With Abortion'],
            range=[PALETTE['cat_2'], PALETTE['accent']]
        ), title=''),
        xOffset='comparison:N',
        tooltip=['cause', 'comparison', 'deaths'],
    ).properties(
        width=1450,
        height=720,
        title={
            'text': f'If Abortion Were a Leading Cause: {race}',
            'subtitle': 'How abortion would rank among top 10 causes of death (2024)',
            'anchor': 'start',
            'offset': 10,
        }
    ).configure_axis(
        labelFontSize=10,
        titleFontSize=11,
        labelColor='#374151',
        titleColor='#374151'
    ).configure_legend(
        labelFontSize=11,
        orient='top',
    )
    
    charts_by_race[race] = chart
    print(f'Chart {2+idx} created')

In [ ]:
# Export race charts
safe_names = {
    'White': 'white',
    'Black or African American': 'black_or_african_american'
}

for idx, race in enumerate(races_to_chart, 1):
    chart = charts_by_race[race]
    safe_name = safe_names[race]
    save_social(chart, cfg, f'0{2+idx}_abortion_comparison_race_{safe_name}', preset='twitter_landscape')
    print(f'✓ Chart {2+idx} ({race}) exported')

## Summary & Cleanup

In [ ]:
conn.close()

# Verify all exports
pngs = sorted(social_dir.glob('*.png'))
print('=== ALL CHARTS COMPLETE ===')
print(f'\n✓ Generated {len(pngs)} publication-ready charts:')
for png in pngs:
    size_kb = png.stat().st_size / 1024
    print(f'  • {png.name} ({size_kb:.0f} KB)')

print('\nAll charts are twitter_landscape (1600×900px) with @unwelcomedata watermark.')
print('Ready for social media posting!')